In [1]:
"""
Análisis Tecno-Económico del Agregador en España
Extracción y Normalización de Perfiles de Demanda (ETL)

Este script procesa las series temporales de demanda estructural basadas en los 
perfiles estándar BDEW (H0: Residencial, G0: Comercial, L0: Agrícola).
Realiza la ingesta de datos cuarto-horarios, filtra el día de estudio, calcula
las medias horarias (resolución del mercado eléctrico) y normaliza los valores
en por unidad (p.u.) para el modelo de optimización MILP.

Autor: Alberto Zafra Muñoz
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional

# ==========================================
# 1. CONSTANTES Y CONFIGURACIÓN
# ==========================================
ARCHIVO_ENTRADA = "Datos_Demanda_P_BASE.csv"
ARCHIVO_SALIDA = "Demanda_P_BASE_Limpios.csv"
FECHA_ESTUDIO = '2016-04-01'

# Mapeo de nomenclaturas técnicas a nombres descriptivos
MAPEO_COLUMNAS = {
    'H0-A_pload': 'U1_Residencial (p.u.)',
    'G0-A_pload': 'U2_Comercial (p.u.)',
    'L0-A_pload': 'U3_Agricola (p.u.)'
}

# ==========================================
# 2. PROCESAMIENTO DE DATOS (ETL)
# ==========================================
def procesar_perfiles_demanda(ruta_csv: str, fecha_objetivo: str) -> Optional[pd.DataFrame]:
    """
    Lee el archivo bruto de demanda, filtra la fecha de estudio y promedia
    los valores a resolución horaria.

    Parámetros:
        ruta_csv (str): Ruta al archivo CSV con los datos de entrada.
        fecha_objetivo (str): Fecha de extracción en formato 'YYYY-MM-DD'.

    Retorna:
        pd.DataFrame: DataFrame con los perfiles horarios promediados.
                      Devuelve None si ocurre un error de lectura.
    """
    print(f"[*] Iniciando extracción de datos desde: {ruta_csv}")
    
    if not os.path.exists(ruta_csv):
        print(f"[ERROR] No se encuentra el archivo fuente '{ruta_csv}'.")
        return None

    try:
        # Carga de datos con formato europeo (separador ';')
        df = pd.read_csv(ruta_csv, sep=";")
        
        # Conversión temporal explícita
        df['time'] = pd.to_datetime(df['time'], format='%d.%m.%Y %H:%M')
        
        # Filtrado del horizonte temporal de estudio
        fecha_filtro = pd.to_datetime(fecha_objetivo).date()
        df_dia = df[df['time'].dt.date == fecha_filtro].copy()

        if df_dia.empty:
            print(f"[ERROR] No hay registros para la fecha {fecha_objetivo}.")
            return None

        print(f"[*] Datos extraídos para el {fecha_objetivo}: {len(df_dia)} registros (15-min).")

        # Extracción de la hora para agrupación
        df_dia['Hora'] = df_dia['time'].dt.hour
        columnas_base = ['Hora'] + list(MAPEO_COLUMNAS.keys())
        
        # Promediado aritmético para pasar de resolución 15-min a horaria
        df_horario = df_dia[columnas_base].groupby('Hora').mean().reset_index()
        df_horario = df_horario.rename(columns=MAPEO_COLUMNAS)

        print("[*] Transformación a resolución horaria completada con éxito.")
        return df_horario

    except Exception as e:
        print(f"[ERROR INESPERADO] Fallo durante el procesamiento: {e}")
        return None

# ==========================================
# 3. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_perfiles_demanda(df: pd.DataFrame):
    """
    Genera la figura de los perfiles de consumo normalizados.
    """
    plt.figure(figsize=(10, 6))
    plt.style.use('seaborn-v0_8-whitegrid')

    # Trazado de los perfiles unitarios
    plt.plot(df['Hora'], df['U1_Residencial (p.u.)'], 
             label='U1: Prosumidor Residencial (Pico Vespertino/Nocturno)', 
             color='#1f77b4', marker='o', linewidth=2.5, markersize=6)
    
    plt.plot(df['Hora'], df['U2_Comercial (p.u.)'], 
             label='U2: Prosumidor Terciario/Oficina (Consumo Diurno)', 
             color='#ff7f0e', marker='s', linewidth=2.5, markersize=6)
    
    plt.plot(df['Hora'], df['U3_Agricola (p.u.)'], 
             label='U3: Prosumidor Agroindustrial (Perfil Constante/Plano)', 
             color='#2ca02c', marker='^', linewidth=2.5, markersize=6)

    # Formateo y Estilos de la Figura
    plt.title(f'Perfiles Normalizados de Demanda Base Inelástica ($P_{{u,t}}^{{BASE}}$)', 
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Hora del día ($t$)', fontsize=12)
    plt.ylabel('Potencia Unitaria (p.u.)', fontsize=12)
    
    plt.xticks(range(0, 24))
    plt.xlim(0, 23)
    plt.ylim(0, df[['U1_Residencial (p.u.)', 'U2_Comercial (p.u.)', 'U3_Agricola (p.u.)']].max().max() * 1.1)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper right', frameon=True, shadow=True, fontsize=10)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    df_resultado = procesar_perfiles_demanda(ARCHIVO_ENTRADA, FECHA_ESTUDIO)
    
    if df_resultado is not None:
        # Guardar dataset limpio como input del optimizador Pyomo
        df_resultado.to_csv(ARCHIVO_SALIDA, index=False)
        print(f"[*] Datos exportados satisfactoriamente a '{ARCHIVO_SALIDA}'.\n")
        
        # Mostrar preview en consola
        print("==================================================================")
        print(" VISTA PREVIA: PERFILES HORARIOS DE DEMANDA BASE (p.u.)")
        print("==================================================================")
        print(df_resultado.to_string(index=False))
        print("==================================================================\n")
        
        print("[*] Generando modelo visual...")
        visualizar_perfiles_demanda(df_resultado)

[*] Iniciando extracción de datos desde: Datos_Demanda_P_BASE.csv
[ERROR] No se encuentra el archivo fuente 'Datos_Demanda_P_BASE.csv'.
